<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/03_classifier_training_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Classifier training

Trains ResNet18 and EfficientNet-B0 (ImageNet-pretrained, fully fine-tuned) as binary tampered/authentic classifiers. All 1,828 spliced images are held out entirely from training and validation — the classifiers learn tampered-vs-authentic from copy-move tampered images plus authentic images only, so the Grad-CAM evaluation in notebook 04 runs on images the classifiers never saw during training.

## Setup — mount Drive, rebuild validated file lists

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import random
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report
from PIL import Image

random.seed(42)
torch.manual_seed(42)

base = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised"  # adjust to your actual path
au_dir = os.path.join(base, "Au")
tp_dir = os.path.join(base, "Tp")

with open(os.path.join(base, "au_list.txt")) as f:
    au_list_content = [line.strip() for line in f if line.strip()]
with open(os.path.join(base, "tp_list.txt")) as f:
    tp_list_content = [line.strip() for line in f if line.strip()]

au_files_final = sorted(set(au_list_content) & set(os.listdir(au_dir)))
tp_files_final = sorted(set(tp_list_content) & set(os.listdir(tp_dir)))

print(f"Authentic: {len(au_files_final)} | Tampered: {len(tp_files_final)}")  # expect 7491, 5123

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Mounted at /content/drive
Authentic: 7491 | Tampered: 5123
Device: cuda


## Train/val split — spliced images fully held out
All 1,828 spliced images are excluded from classifier training and validation entirely. The classifier learns tampered-vs-authentic from copy-move + authentic images only; the full spliced set is reserved for notebook 04's Grad-CAM evaluation. Saved to Drive so it can be reproduced exactly across sessions.

In [2]:
spliced_files_all = [f for f in tp_files_final if f.split("_")[1] == "D"]
non_spliced_tampered = [f for f in tp_files_final if f not in set(spliced_files_all)]
print(f"Spliced (held out): {len(spliced_files_all)} | Non-spliced tampered (train/val pool): {len(non_spliced_tampered)}")

random.shuffle(non_spliced_tampered)
random.shuffle(au_files_final)

def split(files, val_frac=0.2):
    cut = int(len(files) * (1 - val_frac))
    return files[:cut], files[cut:]

train_tp, val_tp = split(non_spliced_tampered)
train_au, val_au = split(au_files_final)

split_dict = {"train_tampered": train_tp, "val_tampered": val_tp,
              "train_authentic": train_au, "val_authentic": val_au,
              "held_out_spliced": spliced_files_all, "seed": 42}
with open("/content/drive/MyDrive/CASIA2.0/casia_split.json", "w") as f:
    json.dump(split_dict, f)

print(f"Train: {len(train_tp)+len(train_au)} | Val: {len(val_tp)+len(val_au)}")

Spliced (held out): 1828 | Non-spliced tampered (train/val pool): 3295
Train: 8628 | Val: 2158


## Leak check — confirm spliced images are disjoint from train/val

In [3]:
assert set(spliced_files_all).isdisjoint(train_tp), "Leak: spliced image in training set"
assert set(spliced_files_all).isdisjoint(val_tp), "Leak: spliced image in validation set"
print("Confirmed: all spliced images are disjoint from classifier train/val.")

Confirmed: all spliced images are disjoint from classifier train/val.


## Copy images to local disk (speeds up training)
Reading thousands of individual files from Drive per epoch is slow. Copying once to Colab's local disk (`/content/`) speeds up every subsequent epoch. Local disk is wiped if the runtime disconnects — re-run this cell if that happens.

In [4]:
import shutil, time

local_base = "/content/CASIA2.0_local"
os.makedirs(local_base, exist_ok=True)

t0 = time.time()
if not os.path.exists(os.path.join(local_base, "Au")):
    shutil.copytree(au_dir, os.path.join(local_base, "Au"))
if not os.path.exists(os.path.join(local_base, "Tp")):
    shutil.copytree(tp_dir, os.path.join(local_base, "Tp"))
print(f"Copied in {time.time()-t0:.1f}s")

au_dir = os.path.join(local_base, "Au")
tp_dir = os.path.join(local_base, "Tp")

Copied in 542.6s


In [5]:
n_local_au = len(os.listdir("/content/CASIA2.0_local/Au"))
n_local_tp = len(os.listdir("/content/CASIA2.0_local/Tp"))

n_drive_au = len(os.listdir("/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised/Au"))
n_drive_tp = len(os.listdir("/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised/Tp"))

print(f"Au: local={n_local_au}, drive={n_drive_au}, match={n_local_au == n_drive_au}")
print(f"Tp: local={n_local_tp}, drive={n_drive_tp}, match={n_local_tp == n_drive_tp}")

Au: local=7492, drive=7492, match=True
Tp: local=5123, drive=5123, match=True


## Dataset, transforms, and DataLoaders
Augmentation (flip, rotation, color jitter) applied only to training data — validation data is left unaugmented so it reflects real performance.

In [6]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CASIADataset(Dataset):
    def __init__(self, tampered_files, authentic_files, tampered_dir, authentic_dir, transform):
        self.samples = [(os.path.join(tampered_dir, f), 1) for f in tampered_files] + \
                        [(os.path.join(authentic_dir, f), 0) for f in authentic_files]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

train_dataset = CASIADataset(train_tp, train_au, tp_dir, au_dir, train_transform)
val_dataset = CASIADataset(val_tp, val_au, tp_dir, au_dir, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 270 | Val batches: 68


## Verify batch-load speed before training

In [7]:
t0 = time.time()
images, labels = next(iter(train_loader))
print(f"First batch loaded in {time.time()-t0:.1f}s")

First batch loaded in 0.6s


## Generic training loop
Shared by both architectures below. Early stopping on validation loss (patience=4) prevents chasing epochs past the point of diminishing returns.

In [9]:
def train_model(model, model_name, max_epochs=30, patience=4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
               "val_f1": [], "val_balanced_acc": []}
    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_labels, all_preds)
        val_bal_acc = balanced_accuracy_score(all_labels, all_preds)

        scheduler.step(val_loss)
        for k, v in zip(history.keys(), [train_loss, val_loss, train_acc, val_acc, val_f1, val_bal_acc]):
            history[k].append(v)

        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
              f"val_acc={val_acc:.4f} val_f1={val_f1:.4f} val_bal_acc={val_bal_acc:.4f}")

        torch.save(model.state_dict(), f"/content/drive/MyDrive/CASIA2.0/{model_name}_last.pt")
        with open(f"/content/drive/MyDrive/CASIA2.0/{model_name}_history.json", "w") as f:
            json.dump(history, f)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), f"/content/drive/MyDrive/CASIA2.0/{model_name}_best.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}.")
                break

    return model, history

## Train ResNet18

In [10]:
resnet = models.resnet18(weights="IMAGENET1K_V1")
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet, resnet_history = train_model(resnet, "casia_resnet")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 150MB/s]


Epoch 1: train_loss=0.5448 val_loss=0.5183 val_acc=0.7215 val_f1=0.4420 val_bal_acc=0.6205
Epoch 2: train_loss=0.4468 val_loss=0.4634 val_acc=0.7465 val_f1=0.6220 val_bal_acc=0.7287
Epoch 3: train_loss=0.4148 val_loss=0.5139 val_acc=0.7359 val_f1=0.5250 val_bal_acc=0.6636
Epoch 4: train_loss=0.3924 val_loss=0.4756 val_acc=0.7475 val_f1=0.6065 val_bal_acc=0.7166
Epoch 5: train_loss=0.3687 val_loss=0.4832 val_acc=0.7595 val_f1=0.6556 val_bal_acc=0.7567
Epoch 6: train_loss=0.3252 val_loss=0.4985 val_acc=0.7521 val_f1=0.6246 val_bal_acc=0.7306
Early stopping at epoch 6.


## Train EfficientNet-B0
Target layer differs (`features[-1]`, not `layer4`) — handled in notebook 04, not here.

In [11]:
effnet = models.efficientnet_b0(weights="IMAGENET1K_V1")
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet, effnet_history = train_model(effnet, "casia_effnet")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 104MB/s]


Epoch 1: train_loss=0.5404 val_loss=0.4804 val_acc=0.7312 val_f1=0.4991 val_bal_acc=0.6492
Epoch 2: train_loss=0.4320 val_loss=0.4438 val_acc=0.7595 val_f1=0.6209 val_bal_acc=0.7274
Epoch 3: train_loss=0.3851 val_loss=0.4435 val_acc=0.7586 val_f1=0.6379 val_bal_acc=0.7412
Epoch 4: train_loss=0.3521 val_loss=0.4537 val_acc=0.7646 val_f1=0.6613 val_bal_acc=0.7613
Epoch 5: train_loss=0.3312 val_loss=0.4793 val_acc=0.7604 val_f1=0.6452 val_bal_acc=0.7472
Epoch 6: train_loss=0.3160 val_loss=0.4897 val_acc=0.7424 val_f1=0.6228 val_bal_acc=0.7295
Epoch 7: train_loss=0.2869 val_loss=0.5143 val_acc=0.7539 val_f1=0.6370 val_bal_acc=0.7408
Early stopping at epoch 7.


## Load existing checkpoints (skip retraining next time)
Loads the best saved checkpoint for each architecture without retraining. Run this instead of the two training cells above once checkpoints exist.

In [12]:
resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_resnet_best.pt", map_location=device))
resnet = resnet.to(device)
resnet.eval()

effnet = models.efficientnet_b0(weights=None)
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_effnet_best.pt", map_location=device))
effnet = effnet.to(device)
effnet.eval()

print("Both checkpoints loaded successfully.")

Both checkpoints loaded successfully.


## Confusion matrix check
Confirms balanced recall across both classes (no majority-class collapse) for whichever model is currently loaded above.

In [14]:
model_to_check = effnet  # swap to effnet to check the other architecture

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_to_check(images)
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["Authentic", "Tampered"]))

[[1178  321]
 [ 200  459]]
              precision    recall  f1-score   support

   Authentic       0.85      0.79      0.82      1499
    Tampered       0.59      0.70      0.64       659

    accuracy                           0.76      2158
   macro avg       0.72      0.74      0.73      2158
weighted avg       0.77      0.76      0.76      2158



In [15]:
import numpy as np

model_to_check.eval()
spliced_preds = []
with torch.no_grad():
    for fname in spliced_files_all[:200]:  # quick sample, not all 1828, just to sanity check
        img = Image.open(os.path.join(tp_dir, fname)).convert("RGB")
        input_tensor = val_transform(img).unsqueeze(0).to(device)
        pred = model_to_check(input_tensor).argmax(1).item()
        spliced_preds.append(pred)

spliced_preds = np.array(spliced_preds)
print(f"Predicted Tampered on spliced sample: {(spliced_preds==1).sum()}/{len(spliced_preds)} "
      f"({(spliced_preds==1).mean()*100:.1f}%)")

Predicted Tampered on spliced sample: 122/200 (61.0%)


In [16]:
spliced_preds_full = []
with torch.no_grad():
    for fname in spliced_files_all:
        img = Image.open(os.path.join(tp_dir, fname)).convert("RGB")
        input_tensor = val_transform(img).unsqueeze(0).to(device)
        pred = model_to_check(input_tensor).argmax(1).item()
        spliced_preds_full.append(pred)

spliced_preds_full = np.array(spliced_preds_full)
print(f"Predicted Tampered on all spliced images: {(spliced_preds_full==1).sum()}/{len(spliced_preds_full)} "
      f"({(spliced_preds_full==1).mean()*100:.1f}%)")

Predicted Tampered on all spliced images: 1074/1828 (58.8%)
